# PulseIQ — Sentiment Fine-Tuning (DistilBERT + LoRA)

Runs on a **Colab T4 runtime** from inside VS Code.

This notebook is a thin driver. All logic lives in tested modules under
`src/pulseiq/training/sentiment/` — the notebook clones the repo and calls them,
so nothing here is untested notebook-only code.

**Order matters:** the zero-shot baseline is scored *before* fine-tuning, on the
same held-out test set, so the before/after comparison is valid.

## 1. Verify the GPU

If this shows no GPU, reconnect and pick a T4 runtime — training on CPU is ~20x slower.

In [2]:
!nvidia-smi

Wed Aug 12 06:56:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo and install

The Colab runtime is a separate machine, so the code has to be fetched.

In [3]:
import os

REPO = "https://github.com/uditnegi16/pulseiq.git"

if not os.path.exists("pulseiq"):
    !git clone -q $REPO
%cd pulseiq
!git pull -q

!pip install -q transformers peft datasets accelerate evaluate scikit-learn pyarrow pydantic-settings mlflow
print("\ninstalled")

/content/pulseiq

installed


In [4]:
import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")

import torch
print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
cuda : True
gpu  : Tesla T4


## 3. Build the dataset

Streams Amazon Reviews'23 (Electronics), maps stars to binary sentiment,
balances the classes, and writes train/val/test to parquet.

**Balancing matters:** ~80% of Amazon reviews are 4–5 stars. Without it, a model
that always predicts "positive" would score 80% and appear to work.

**Labels are a proxy.** A 5-star review can still contain complaints, so the
accuracy ceiling is set by label noise, not model capacity. See
`docs/decision-log.md` D-007.

In [5]:
from pulseiq.training.sentiment.dataset import load_reviews, stratified_split, save_splits
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

MAX_ROWS = 5_000     # raise to 20_000 once the pipeline is proven

# The canonical McAuley-Lab repo serves raw_review_* through a loading script,
# which datasets>=4.0 refuses to execute. The loader tries Parquet mirrors in
# order and reports which one it used.
frame = load_reviews(max_rows=MAX_ROWS, balance=True)

print(f"\n{len(frame)} labelled reviews")
print(f"positive share: {frame['label'].mean():.1%}")
print(f"median length : {frame['text'].str.split().str.len().median():.0f} words")
frame.head(3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]


5000 labelled reviews
positive share: 50.0%
median length : 43 words


,text,label,rating,verified_purchase,timestamp
0,Didn’t work at all lenses loose/broken. These ...,0,1.0,None,None
1,Excellent! I love these. They even come with a...,1,5.0,None,None
2,Great laptop backpack! I was searching for a s...,1,5.0,None,None


In [6]:
train, val, test = stratified_split(frame, test_size=0.15, val_size=0.15)
paths = save_splits(train, val, test, Path("data/processed"))
print(f"train={len(train)} val={len(val)} test={len(test)}")

train=3500 val=750 test=750


## 4. Zero-shot baseline — the "before" number

`distilbert-base-uncased-finetuned-sst-2-english`, used off the shelf. This is
what the original project did.

It is a **strong** baseline — already sentiment-tuned, just on movie reviews
rather than product reviews. A weak baseline chosen to flatter the fine-tune
would make the improvement meaningless.

In [7]:
from pulseiq.training.sentiment.baseline_zeroshot import evaluate_baseline
from pulseiq.evaluation.classification import to_metrics, majority_baseline
import json

baseline = evaluate_baseline(test, device=0 if torch.cuda.is_available() else -1)

print("\nzero-shot baseline")
print(to_metrics(baseline))
print()
print(to_metrics(baseline).confusion_table())
print("\nmajority-class floor:", to_metrics(majority_baseline(test['label'])))

Path("reports").mkdir(exist_ok=True)
Path("reports/sentiment_baseline.json").write_text(json.dumps(baseline, indent=2))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



zero-shot baseline
acc=0.8880 f1=0.8797 macro_f1=0.8875 prec=0.9505 rec=0.8187 n=750

              pred neg  pred pos
    true neg       359        16
    true pos        68       307

majority-class floor: acc=0.5000 f1=0.0000 macro_f1=0.3333 prec=0.0000 rec=0.0000 n=750


264

## 5. Fine-tune with LoRA

LoRA freezes DistilBERT and trains small low-rank matrices in the attention
layers — under 1% of parameters, and a few MB on disk.

**Checkpoint selection uses the validation set.** The test set is scored exactly
once, at the end. Selecting on test would be the classification equivalent of
the temporal leakage this project fixed in forecasting.

In [1]:
!pip install -q --upgrade "torchao>=0.16.0"

# peft's LoRA dispatcher calls is_torchao_available() unconditionally and raises
# on Colab's preinstalled torchao 0.10.0. Upgrading satisfies the version check.
import importlib, torchao
importlib.reload(torchao)
print("torchao:", torchao.__version__)

torchao: 0.18.0


In [8]:
from pulseiq.training.sentiment.dataset import stratified_split
from pulseiq.training.sentiment.finetune_lora import LoRAConfig
from pulseiq.training.sentiment.finetune_lora import train as run_training
from pulseiq.training.sentiment.finetune_lora import evaluate_on_test

train_df, val_df, test_df = stratified_split(frame, test_size=0.15, val_size=0.15, seed=42)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

config = LoRAConfig(
    r=16,
    lora_alpha=32,
    learning_rate=2e-4,
    batch_size=32,
    epochs=3,
    max_length=256,
)

model, tokenizer, run_info = run_training(train_df, val_df, config=config)
print("\n", {k: v for k, v in run_info.items() if k in
      ("device", "train_runtime_seconds", "trainable_params", "total_params", "adapter_size_mb")})

train=3500 val=750 test=750


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1,Macro F1
1,0.231300,0.183602,0.934667,0.935948,0.934641
2,0.174805,0.172757,0.940000,0.940711,0.939991
3,0.152170,0.161975,0.942667,0.942436,0.942666



 {'device': 'cuda', 'train_runtime_seconds': 60.865, 'adapter_size_mb': 4.27, 'trainable_params': 887042, 'total_params': 67842052}


## 6. Score on the held-out test set

First and only time the test set is touched by the fine-tuned model.

In [10]:
finetuned = evaluate_on_test(model, tokenizer, test_df, max_length=config.max_length)

print("fine-tuned (LoRA)")
print(to_metrics(finetuned))
print()
print(to_metrics(finetuned).confusion_table())

Path("reports/sentiment_finetuned.json").write_text(json.dumps({**run_info, **finetuned}, indent=2))

fine-tuned (LoRA)
acc=0.9413 f1=0.9409 macro_f1=0.9413 prec=0.9485 rec=0.9333 n=750

              pred neg  pred pos
    true neg       356        19
    true pos        25       350


686

## 7. Before vs after

`error_reduction_pct` is the honest framing at high accuracy: +3 points from 91% to 94% removes a third of the remaining errors.

In [11]:
from pulseiq.evaluation.classification import compare
import pandas as pd

comparison = pd.DataFrame({
    "zero-shot": {k: baseline[k] for k in ("accuracy","precision","recall","f1","macro_f1")},
    "fine-tuned": {k: finetuned[k] for k in ("accuracy","precision","recall","f1","macro_f1")},
})
comparison["delta"] = comparison["fine-tuned"] - comparison["zero-shot"]
print(comparison.round(4))

print()
for key, value in compare(baseline, finetuned).items():
    print(f"  {key:<22}: {value:+.4f}")

           zero-shot  fine-tuned   delta
accuracy      0.8880      0.9413  0.0533
precision     0.9505      0.9485 -0.0020
recall        0.8187      0.9333  0.1147
f1            0.8797      0.9409  0.0612
macro_f1      0.8875      0.9413  0.0539

  accuracy_delta        : +0.0533
  f1_delta              : +0.0612
  macro_f1_delta        : +0.0539
  precision_delta       : -0.0020
  recall_delta          : +0.1147
  error_reduction_pct   : +47.6190


## 8. Download the adapter

The adapter is a few MB, small enough to commit. Download it, place it in
`models/sentiment_lora/` locally, and inference runs on CPU with no GPU needed.

In [13]:
import shutil
shutil.make_archive("sentiment_lora", "zip", "models/sentiment_lora")
size_mb = os.path.getsize("sentiment_lora.zip") / 1e6
print(f"sentiment_lora.zip — {size_mb:.2f} MB")

try:
    from google.colab import files
    files.download("sentiment_lora.zip")
except ImportError:
    print("Not in the Colab web UI — right-click the file in the VS Code explorer to download.")

sentiment_lora.zip — 3.50 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import subprocess, os

# Confirm it exists on the remote runtime
print("zip at:", os.path.abspath("sentiment_lora.zip"),
      f"({os.path.getsize('sentiment_lora.zip')/1e6:.2f} MB)")

# Allow the adapter through .gitignore (models/* is blocked by default)
gitignore = open(".gitignore").read()
if "!models/sentiment_lora/" not in gitignore:
    with open(".gitignore", "a") as f:
        f.write("\n# Fine-tuned LoRA adapter - small enough to commit, and\n"
                "# committing it is what makes the result reproducible.\n"
                "!models/sentiment_lora/\n!models/sentiment_lora/**\n")

!git config user.email "uditnegi16@users.noreply.github.com"
!git config user.name "uditnegi16"
!git add -f models/sentiment_lora .gitignore
!git commit -q -m "Add fine-tuned LoRA adapter (4.27 MB): 94.1% accuracy, 47.6% error reduction"
print("committed — now push with a token")

zip at: /content/pulseiq/sentiment_lora.zip (3.50 MB)
committed — now push with a token


In [20]:
import subprocess

def sh(*args):
    return subprocess.run(args, capture_output=True, text=True).stdout

print(sh("git", "log", "--oneline", "-3"))
print("--- status ---")
print(sh("git", "status", "--short") or "(clean)")
print("--- adapter tracked? ---")
print(sh("git", "ls-files", "models/sentiment_lora") or "NOT TRACKED")
print("--- files on disk ---")
print(sh("ls", "-la", "models/sentiment_lora"))

17a70e0 Add fine-tuned LoRA adapter (4.27 MB): 94.1% accuracy, 47.6% error reduction
0c0df24 Docs: error log E-007 to E-009, decision log D-012 to D-016, new outcome log
2c57148 Fix dataset loading: datasets>=4.0 removed trust_remote_code; add Parquet mirror fallback

--- status ---
?? sentiment_lora.zip

--- adapter tracked? ---
models/sentiment_lora/README.md
models/sentiment_lora/adapter_config.json
models/sentiment_lora/adapter_model.safetensors
models/sentiment_lora/tokenizer.json
models/sentiment_lora/tokenizer_config.json

--- files on disk ---
total 4192
drwxr-xr-x 2 root root    4096 Aug 12 06:58 .
drwxr-xr-x 4 root root    4096 Aug 12 06:58 ..
-rw-r--r-- 1 root root    1051 Aug 12 06:58 adapter_config.json
-rw------- 1 root root 3552056 Aug 12 06:58 adapter_model.safetensors
-rw-r--r-- 1 root root    5170 Aug 12 06:58 README.md
-rw-r--r-- 1 root root     351 Aug 12 06:58 tokenizer_config.json
-rw-r--r-- 1 root root  711494 Aug 12 06:58 tokenizer.json



In [21]:
import subprocess
from getpass import getpass

token = getpass("GitHub token: ")

result = subprocess.run(
    ["git", "push", f"https://{token}@github.com/uditnegi16/pulseiq.git", "main"],
    capture_output=True, text=True,
)
print("exit code:", result.returncode)
print(result.stderr.replace(token, "***") if result.stderr else "")
print(result.stdout.replace(token, "***") if result.stdout else "")

exit code: 0
Everything up-to-date




In [22]:
import subprocess
def sh(*a): return subprocess.run(a, capture_output=True, text=True).stdout

print("branch:", sh("git", "rev-parse", "--abbrev-ref", "HEAD").strip())
print("local HEAD :", sh("git", "rev-parse", "HEAD").strip()[:8])
print("remote main:", sh("git", "ls-remote", "origin", "main").split()[0][:8] if sh("git","ls-remote","origin","main") else "unreachable")

branch: main
local HEAD : 17a70e01
remote main: 17a70e01


In [23]:
git pull
Get-ChildItem models\sentiment_lora

SyntaxError: invalid syntax (77687020.py, line 1)

In [19]:
import subprocess
from getpass import getpass

token = getpass("GitHub token: ")

result = subprocess.run(
    ["git", "push", f"https://{token}@github.com/uditnegi16/pulseiq.git", "main"],
    capture_output=True, text=True,
)
# Never print the raw output: it can echo the URL, token included.
print("exit code:", result.returncode)
print(result.stderr.replace(token, "***") if result.stderr else "")
print(result.stdout.replace(token, "***") if result.stdout else "")

exit code: 0
Everything up-to-date




## 9. Log to MLflow (optional)

Only works if the runtime can reach your tracking store. Skip if MLflow is
local-only — the JSON reports in `reports/` carry the numbers either way.

In [ ]:
# import mlflow
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
# mlflow.set_experiment("pulseiq")
#
# with mlflow.start_run(run_name="sentiment_lora_finetune"):
#     mlflow.log_params({**config.as_dict(), "n_train": len(train), "n_test": len(test)})
#     mlflow.log_metrics({k: v for k, v in finetuned.items() if isinstance(v, (int, float))})
#     mlflow.log_metrics({f"baseline_{k}": v for k, v in baseline.items() if isinstance(v, (int, float))})

---

## Next steps

1. Download `sentiment_lora.zip`, unzip into `models/sentiment_lora/` locally
2. Verify CPU inference: `python -m pulseiq.training.sentiment.predict --text "battery died in a week"`
3. Copy the before/after table into `docs/metrics.md`
4. Commit the adapter (a few MB) so the result is reproducible from the repo

**When writing up:** report the majority-class floor alongside the accuracy, and
state that labels are star-derived proxies. A fine-tune that reaches ~92% where
labels are ~95% faithful has saturated the task — pushing further would be
fitting label noise.